# Notebook 07 — Results Consolidation, Robustness, and Hypothesis Evaluation

This notebook consolidates the completed outputs from Notebooks 03–06 into one reproducible results package.

It performs five tasks:

1. **RQ1 / H1 timestamp-fidelity audit** — compares aircraft event-time timestamps with snapshot/processing-time timestamps using mean absolute error (MAE).
2. **Model comparison** — combines the strongest chronological fused tabular baseline, the fused temporal Transformer, and both R-GCN variants.
3. **Paired bootstrap uncertainty** — estimates 95% confidence intervals for test AUPRC and paired AUPRC differences on the same held-out events.
4. **Robustness and sensitivity** — summarizes temporal-window sensitivity and subgroup performance without changing frozen thresholds.
5. **Explainability synthesis** — consolidates Transformer attribution, R-GCN node-feature attribution, and R-GCN relation-weight diagnostics.

### Frozen hypothesis criteria

- **H1:** Event-time timestamp alignment reduces MAE by at least **1.0 second** relative to processing-time timestamps.
- **H2:** Full R-GCN test AUPRC exceeds temporal Transformer test AUPRC by at least **0.03**.
- **H3:** Lag/relation-aware R-GCN test AUPRC exceeds the lag/relation ablation by at least **0.05**, while local p95 latency remains at or below **500 ms**.

> **Interpretation boundary:** The aircraft and NF-UNSW-NB15-v3 data are independent datasets. Notebook 03 uses a controlled temporal mapping when their original UTC ranges do not overlap. Results therefore quantify behavior within the modeled alignment design and do not establish causal or site-coincident aircraft/cyber incidents.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from numbers import Integral, Real
from typing import Any, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

RESULTS_VERSION = "results_v2"
RANDOM_STATE = 8499
BOOTSTRAP_ITERATIONS = int(os.environ.get("RESULTS_BOOTSTRAP_ITERATIONS", "2000"))
MIN_SUBGROUP_ROWS = int(os.environ.get("RESULTS_MIN_SUBGROUP_ROWS", "25"))
H1_REQUIRED_MAE_REDUCTION_S = 1.0
H2_REQUIRED_AUPRC_MARGIN = 0.03
H3_REQUIRED_AUPRC_GAIN = 0.05
H3_MAX_P95_LATENCY_MS = 500.0
PRIMARY_WINDOW_HALF_WIDTH_S = 120


def locate_project_root(start: Path) -> Path:
    required = [
        Path("data/processed/point_event_v2/modeling_v4/baseline_model_results_v4.csv"),
        Path("data/processed/point_event_v2/transformer_v2/transformer_reference_v2.json"),
        Path("data/processed/point_event_v2/rgcn_leakage_safe_v2/rgcn_reference_v2.json"),
    ]
    for candidate in [start.resolve(), *start.resolve().parents]:
        if all((candidate / relative).is_file() for relative in required):
            return candidate
    return start.resolve()


PROJECT_ROOT = locate_project_root(Path.cwd())
BASE_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2"
ALIGNMENT_DIR = BASE_DIR / "nf3_alignment_v2"
MODELING_DIR = BASE_DIR / "modeling_v4"
TRANSFORMER_DIR = BASE_DIR / "transformer_v2"
RGCN_DIR = BASE_DIR / "rgcn_leakage_safe_v2"
OUTPUT_DIR = BASE_DIR / RESULTS_VERSION
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FUSED_IN = ALIGNMENT_DIR / "candidate_point_events_v2_with_context_and_nf3_primary.csv"
WINDOW_FEATURES_IN = ALIGNMENT_DIR / "event_nf3_window_features_v2.csv"
WINDOW_TARGETS_IN = ALIGNMENT_DIR / "event_nf3_targets_v2.csv"
ALIGNMENT_SUMMARY_IN = ALIGNMENT_DIR / "point_event_nf3_alignment_leakage_safe_v2_build_summary.json"

SPLITS_IN = MODELING_DIR / "model_split_assignments_v4.csv"
TABULAR_RESULTS_IN = MODELING_DIR / "baseline_model_results_v4.csv"
TABULAR_PREDICTIONS_IN = MODELING_DIR / "baseline_test_predictions_v4.csv"
TABULAR_TARGET_DEFINITION_IN = MODELING_DIR / "model_target_definition_v4.json"

TRANSFORMER_RESULTS_IN = TRANSFORMER_DIR / "temporal_transformer_results_v2.csv"
TRANSFORMER_PREDICTIONS_IN = TRANSFORMER_DIR / "temporal_transformer_predictions_v2.csv"
TRANSFORMER_REFERENCE_IN = TRANSFORMER_DIR / "transformer_reference_v2.json"
TRANSFORMER_ATTRIBUTION_IN = TRANSFORMER_DIR / "temporal_transformer_feature_attribution_v2.csv"
TRANSFORMER_ATTENTION_IN = TRANSFORMER_DIR / "temporal_transformer_attention_summary_v2.csv"

RGCN_RESULTS_IN = RGCN_DIR / "rgcn_results_v2.csv"
RGCN_PREDICTIONS_IN = RGCN_DIR / "rgcn_predictions_v2.csv"
RGCN_REFERENCE_IN = RGCN_DIR / "rgcn_reference_v2.json"
RGCN_HYPOTHESES_IN = RGCN_DIR / "rgcn_hypothesis_evaluation_v2.csv"
RGCN_ATTRIBUTION_IN = RGCN_DIR / "rgcn_feature_attribution_v2.csv"
RGCN_RELATION_NORMS_IN = RGCN_DIR / "rgcn_relation_weight_norms_v2.csv"
RGCN_RELATION_STATS_IN = RGCN_DIR / "rgcn_relation_statistics_v2.csv"

RQ1_MAE_OUT = OUTPUT_DIR / "rq1_timestamp_alignment_mae_v2.csv"
RQ1_BOOTSTRAP_OUT = OUTPUT_DIR / "rq1_timestamp_alignment_bootstrap_v2.csv"
WINDOW_SENSITIVITY_OUT = OUTPUT_DIR / "alignment_window_sensitivity_v2.csv"
MODEL_COMPARISON_OUT = OUTPUT_DIR / "model_comparison_v2.csv"
PAIRED_PREDICTIONS_OUT = OUTPUT_DIR / "paired_test_prediction_bank_v2.csv"
MODEL_BOOTSTRAP_OUT = OUTPUT_DIR / "model_auprc_bootstrap_v2.csv"
PAIRED_DIFFERENCES_OUT = OUTPUT_DIR / "paired_model_auprc_differences_v2.csv"
SUBGROUP_OUT = OUTPUT_DIR / "subgroup_performance_v2.csv"
EXPLAINABILITY_OUT = OUTPUT_DIR / "explainability_summary_v2.csv"
HYPOTHESES_OUT = OUTPUT_DIR / "consolidated_hypothesis_evaluation_v2.csv"
INTEGRITY_OUT = OUTPUT_DIR / "results_integrity_checks_v2.csv"
REFERENCE_OUT = OUTPUT_DIR / "results_reference_v2.json"
SUMMARY_OUT = OUTPUT_DIR / "results_build_summary_v2.json"
NARRATIVE_OUT = OUTPUT_DIR / "results_narrative_v2.md"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "results_data_dictionary_v2.csv"
CONFIG_OUT = OUTPUT_DIR / "results_config_v2.json"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

print("Project root:", PROJECT_ROOT)
print("Results output:", OUTPUT_DIR)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)

In [ ]:
def require_text(value: object, name: str) -> str:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, bytes):
        text = value.decode("utf-8", errors="strict")
    else:
        text = str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{name} is empty or missing.")
    return text


def require_float(value: object, name: str) -> float:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, (Real, np.number)):
        converted = float(value)
    else:
        text = require_text(value, name)
        try:
            converted = float(text)
        except ValueError as exc:
            raise ValueError(f"{name} is not numeric: {value!r}") from exc
    if not math.isfinite(converted):
        raise ValueError(f"{name} is not finite: {value!r}")
    return converted


def optional_float(value: object) -> float | None:
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    try:
        converted = float(str(value))
    except (TypeError, ValueError):
        return None
    return converted if math.isfinite(converted) else None


def require_int(value: object, name: str) -> int:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (Integral, np.integer)):
        return int(value)
    if isinstance(value, (Real, np.floating)):
        converted = float(value)
        if not math.isfinite(converted) or not converted.is_integer():
            raise ValueError(f"{name} is not an integer: {value!r}")
        return int(converted)
    text = require_text(value, name)
    try:
        converted = float(text)
    except ValueError as exc:
        raise ValueError(f"{name} is not integer-compatible: {value!r}") from exc
    if not math.isfinite(converted) or not converted.is_integer():
        raise ValueError(f"{name} is not an integer: {value!r}")
    return int(converted)


def require_columns(frame: pd.DataFrame, columns: Iterable[str], table_name: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def parse_utc_series(series: pd.Series, name: str) -> pd.Series:
    parsed = pd.to_datetime(series.astype("string"), utc=True, errors="coerce")
    if parsed.isna().any():
        bad_count = require_int(parsed.isna().sum(), f"invalid {name} count")
        raise ValueError(f"{name} contains {bad_count:,} invalid timestamp values.")
    return parsed


def read_json(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {path}")
    return payload


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def slugify(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def has_both_classes(values: object) -> bool:
    if isinstance(values, pd.Series):
        raw_values = values.to_numpy(dtype=object, copy=False)
    else:
        raw_values = np.asarray(values, dtype=object)

    numeric_values = pd.to_numeric(
        raw_values,
        errors="coerce",
    )
    numeric_array = np.asarray(numeric_values, dtype=float)
    finite_values = numeric_array[np.isfinite(numeric_array)]
    if finite_values.size == 0:
        return False
    return np.unique(finite_values.astype(np.int8, copy=False)).size == 2


def safe_auprc(target: np.ndarray, probability: np.ndarray) -> float:
    if np.unique(target).size < 2:
        return np.nan
    return float(average_precision_score(target, probability))


def safe_roc_auc(target: np.ndarray, probability: np.ndarray) -> float:
    if np.unique(target).size < 2:
        return np.nan
    return float(roc_auc_score(target, probability))


def classification_metrics(
    target: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> dict[str, float | int]:
    prediction = (probability >= threshold).astype(np.int8)
    matrix = confusion_matrix(target, prediction, labels=[0, 1])
    return {
        "auprc": safe_auprc(target, probability),
        "roc_auc": safe_roc_auc(target, probability),
        "precision": float(precision_score(target, prediction, zero_division=0)),
        "recall": float(recall_score(target, prediction, zero_division=0)),
        "f1": float(f1_score(target, prediction, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(target, prediction)),
        "brier_score": float(brier_score_loss(target, probability)),
        "tn": require_int(matrix[0, 0], "true negatives"),
        "fp": require_int(matrix[0, 1], "false positives"),
        "fn": require_int(matrix[1, 0], "false negatives"),
        "tp": require_int(matrix[1, 1], "true positives"),
    }


def stratified_bootstrap_positions(target: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    negative_positions = np.flatnonzero(target == 0)
    positive_positions = np.flatnonzero(target == 1)
    if negative_positions.size == 0 or positive_positions.size == 0:
        raise ValueError("Stratified bootstrap requires both target classes.")
    negative_sample = rng.choice(negative_positions, size=negative_positions.size, replace=True)
    positive_sample = rng.choice(positive_positions, size=positive_positions.size, replace=True)
    sampled = np.concatenate([negative_sample, positive_sample])
    rng.shuffle(sampled)
    return sampled.astype(np.int64, copy=False)


def percentile_interval(
    values: object,
    lower: float = 2.5,
    upper: float = 97.5,
) -> tuple[float, float]:
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return np.nan, np.nan
    return float(np.percentile(array, lower)), float(np.percentile(array, upper))


def bootstrap_mean(values: np.ndarray, iterations: int, seed: int) -> np.ndarray:
    clean = np.asarray(values, dtype=float)
    clean = clean[np.isfinite(clean)]
    if clean.size == 0:
        return np.asarray([], dtype=float)
    rng = np.random.default_rng(seed)
    estimates = np.empty(iterations, dtype=float)
    for iteration in range(iterations):
        sample = rng.choice(clean, size=clean.size, replace=True)
        estimates[iteration] = float(np.mean(sample))
    return estimates


def bootstrap_model_auprc(
    target: np.ndarray,
    probabilities: Mapping[str, np.ndarray],
    iterations: int,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    rng = np.random.default_rng(seed)
    distributions = {
        model_key: np.empty(iterations, dtype=float)
        for model_key in probabilities
    }
    for iteration in range(iterations):
        positions = stratified_bootstrap_positions(target, rng)
        sampled_target = target[positions]
        for model_key, model_probability in probabilities.items():
            distributions[model_key][iteration] = safe_auprc(
                sampled_target,
                model_probability[positions],
            )
    rows: list[dict[str, object]] = []
    for model_key, distribution in distributions.items():
        lower, upper = percentile_interval(distribution)
        rows.append(
            {
                "model_key": model_key,
                "point_test_auprc": safe_auprc(target, probabilities[model_key]),
                "bootstrap_mean_auprc": float(np.mean(distribution)),
                "bootstrap_ci_lower_95": lower,
                "bootstrap_ci_upper_95": upper,
                "bootstrap_iterations": iterations,
            }
        )
    return pd.DataFrame(rows), distributions


def paired_difference_summary(
    left_key: str,
    right_key: str,
    distributions: Mapping[str, np.ndarray],
    point_estimates: Mapping[str, float],
    required_margin: float,
    comparison_name: str,
) -> dict[str, object]:
    difference = distributions[left_key] - distributions[right_key]
    lower, upper = percentile_interval(difference)
    point_difference = point_estimates[left_key] - point_estimates[right_key]
    return {
        "comparison": comparison_name,
        "left_model_key": left_key,
        "right_model_key": right_key,
        "point_auprc_difference": point_difference,
        "bootstrap_mean_difference": float(np.mean(difference)),
        "bootstrap_ci_lower_95": lower,
        "bootstrap_ci_upper_95": upper,
        "probability_difference_gt_zero": float(np.mean(difference > 0.0)),
        "required_margin": required_margin,
        "probability_meets_required_margin": float(np.mean(difference >= required_margin)),
        "point_margin_pass": point_difference >= required_margin,
    }


print("Helper functions loaded.")

In [ ]:
required_files = [
    FUSED_IN,
    WINDOW_FEATURES_IN,
    WINDOW_TARGETS_IN,
    SPLITS_IN,
    TABULAR_RESULTS_IN,
    TABULAR_PREDICTIONS_IN,
    TABULAR_TARGET_DEFINITION_IN,
    TRANSFORMER_RESULTS_IN,
    TRANSFORMER_PREDICTIONS_IN,
    TRANSFORMER_REFERENCE_IN,
    RGCN_RESULTS_IN,
    RGCN_PREDICTIONS_IN,
    RGCN_REFERENCE_IN,
    RGCN_HYPOTHESES_IN,
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(
        "Notebook 07 requires completed outputs from Notebooks 03–06. Missing:\n"
        + formatted
    )

fused = pd.read_csv(FUSED_IN, low_memory=False)
window_features = pd.read_csv(WINDOW_FEATURES_IN, low_memory=False)
window_targets = pd.read_csv(WINDOW_TARGETS_IN, low_memory=False)
splits = pd.read_csv(SPLITS_IN, low_memory=False)
tabular_results = pd.read_csv(TABULAR_RESULTS_IN, low_memory=False)
tabular_predictions = pd.read_csv(TABULAR_PREDICTIONS_IN, low_memory=False)
transformer_results = pd.read_csv(TRANSFORMER_RESULTS_IN, low_memory=False)
transformer_predictions = pd.read_csv(TRANSFORMER_PREDICTIONS_IN, low_memory=False)
rgcn_results = pd.read_csv(RGCN_RESULTS_IN, low_memory=False)
rgcn_predictions = pd.read_csv(RGCN_PREDICTIONS_IN, low_memory=False)
rgcn_hypotheses = pd.read_csv(RGCN_HYPOTHESES_IN, low_memory=False)

target_definition = read_json(TABULAR_TARGET_DEFINITION_IN)
transformer_reference = read_json(TRANSFORMER_REFERENCE_IN)
rgcn_reference = read_json(RGCN_REFERENCE_IN)
alignment_summary = read_json(ALIGNMENT_SUMMARY_IN) if ALIGNMENT_SUMMARY_IN.is_file() else {}

optional_tables: dict[str, pd.DataFrame] = {}
for table_name, path in {
    "transformer_attribution": TRANSFORMER_ATTRIBUTION_IN,
    "transformer_attention": TRANSFORMER_ATTENTION_IN,
    "rgcn_attribution": RGCN_ATTRIBUTION_IN,
    "rgcn_relation_norms": RGCN_RELATION_NORMS_IN,
    "rgcn_relation_stats": RGCN_RELATION_STATS_IN,
}.items():
    optional_tables[table_name] = (
        pd.read_csv(path, low_memory=False) if path.is_file() else pd.DataFrame()
    )

loaded_shapes = pd.DataFrame(
    [
        {"table": "fused", "rows": len(fused), "columns": len(fused.columns)},
        {"table": "window_features", "rows": len(window_features), "columns": len(window_features.columns)},
        {"table": "window_targets", "rows": len(window_targets), "columns": len(window_targets.columns)},
        {"table": "splits", "rows": len(splits), "columns": len(splits.columns)},
        {"table": "tabular_results", "rows": len(tabular_results), "columns": len(tabular_results.columns)},
        {"table": "tabular_predictions", "rows": len(tabular_predictions), "columns": len(tabular_predictions.columns)},
        {"table": "transformer_results", "rows": len(transformer_results), "columns": len(transformer_results.columns)},
        {"table": "transformer_predictions", "rows": len(transformer_predictions), "columns": len(transformer_predictions.columns)},
        {"table": "rgcn_results", "rows": len(rgcn_results), "columns": len(rgcn_results.columns)},
        {"table": "rgcn_predictions", "rows": len(rgcn_predictions), "columns": len(rgcn_predictions.columns)},
    ]
)
display(loaded_shapes)

In [ ]:
require_columns(
    fused,
    ["event_id", "aircraft_event_time_utc", "snapshot_time_utc", "event_lag_seconds"],
    "primary fused event table",
)
require_columns(
    splits,
    ["event_id", "model_target", "chronological_split"],
    "model split assignments",
)

fused["event_id"] = fused["event_id"].astype("string")
splits["event_id"] = splits["event_id"].astype("string")
fused["aircraft_event_time_utc"] = parse_utc_series(
    fused["aircraft_event_time_utc"],
    "aircraft_event_time_utc",
)
fused["snapshot_time_utc"] = parse_utc_series(
    fused["snapshot_time_utc"],
    "snapshot_time_utc",
)

integrity_rows: list[dict[str, object]] = []

def add_check(name: str, expected: object, actual: object, match: bool, severity: str = "error") -> None:
    integrity_rows.append(
        {
            "check": name,
            "expected": expected,
            "actual": actual,
            "match": bool(match),
            "severity": severity,
        }
    )

add_check("fused_event_id_unique", True, fused["event_id"].is_unique, fused["event_id"].is_unique)
add_check("split_event_id_unique", True, splits["event_id"].is_unique, splits["event_id"].is_unique)
add_check("fused_rows_equal_split_rows", len(fused), len(splits), len(fused) == len(splits))
add_check(
    "fused_and_split_event_sets_equal",
    True,
    set(fused["event_id"]) == set(splits["event_id"]),
    set(fused["event_id"]) == set(splits["event_id"]),
)
add_check(
    "chronological_split_values_valid",
    {"train", "validation", "test"},
    sorted(splits["chronological_split"].dropna().astype(str).unique().tolist()),
    set(splits["chronological_split"].dropna().astype(str).unique()) <= {"train", "validation", "test"},
)
add_check(
    "model_target_binary",
    {0, 1},
    sorted(pd.to_numeric(splits["model_target"], errors="coerce").dropna().astype(int).unique().tolist()),
    set(pd.to_numeric(splits["model_target"], errors="coerce").dropna().astype(int).unique()) <= {0, 1},
)

integrity_checks = pd.DataFrame(integrity_rows)
integrity_checks.to_csv(INTEGRITY_OUT, index=False)
display(integrity_checks)

failed_errors = integrity_checks.loc[
    integrity_checks["severity"].eq("error") & ~integrity_checks["match"]
]
if not failed_errors.empty:
    raise AssertionError("One or more structural integrity checks failed.")

model_data = fused.merge(
    splits[["event_id", "model_target", "chronological_split"]],
    on="event_id",
    how="inner",
    validate="one_to_one",
)
model_data["model_target"] = pd.to_numeric(
    model_data["model_target"], errors="raise"
).astype("int8")

split_summary = (
    model_data.groupby("chronological_split", sort=False)
    .agg(
        rows=("event_id", "size"),
        positives=("model_target", "sum"),
        prevalence=("model_target", "mean"),
        first_event_time_utc=("aircraft_event_time_utc", "min"),
        last_event_time_utc=("aircraft_event_time_utc", "max"),
    )
    .reset_index()
)
display(split_summary)

## RQ1 / H1 — Event-Time Versus Processing-Time Timestamp Fidelity

The aircraft-reported event timestamp is treated as the physical event-time reference. The snapshot timestamp represents when the observation was collected or processed. Therefore:

- event-time absolute error relative to the event-time reference is zero by definition;
- processing-time absolute error is `|snapshot_time_utc − aircraft_event_time_utc|`;
- the paired MAE reduction is the mean processing-time error.

This is a timestamp-fidelity test. It is **not** evidence of real aircraft/network clock synchronization because the cyber data are independently sourced.

In [ ]:
alignment_audit = model_data[
    [
        "event_id",
        "aircraft_event_time_utc",
        "snapshot_time_utc",
        "event_lag_seconds",
        *(["stale_event_flag"] if "stale_event_flag" in model_data.columns else []),
    ]
].copy()

alignment_audit["event_time_absolute_error_s"] = 0.0
alignment_audit["processing_time_absolute_error_s"] = (
    alignment_audit["snapshot_time_utc"] - alignment_audit["aircraft_event_time_utc"]
).dt.total_seconds().abs()
alignment_audit["reported_event_lag_absolute_s"] = pd.to_numeric(
    alignment_audit["event_lag_seconds"], errors="coerce"
).abs()

lag_consistency = np.isclose(
    alignment_audit["processing_time_absolute_error_s"].to_numpy(dtype=float),
    alignment_audit["reported_event_lag_absolute_s"].to_numpy(dtype=float),
    equal_nan=False,
    atol=1e-6,
)
if not bool(np.all(lag_consistency)):
    mismatch_count = require_int((~lag_consistency).sum(), "event-lag mismatch count")
    raise AssertionError(
        f"Computed processing-time errors disagree with event_lag_seconds for {mismatch_count:,} rows."
    )

alignment_subsets: dict[str, pd.Series] = {
    "all_events": pd.Series(True, index=alignment_audit.index, dtype=bool),
}
if "stale_event_flag" in alignment_audit.columns:
    stale_numeric = pd.to_numeric(
        alignment_audit["stale_event_flag"], errors="coerce"
    ).fillna(0)
    alignment_subsets["non_stale_events"] = stale_numeric.eq(0)
    alignment_subsets["stale_events"] = stale_numeric.eq(1)

rq1_rows: list[dict[str, object]] = []
for subset_name, subset_mask in alignment_subsets.items():
    subset = alignment_audit.loc[subset_mask]
    if subset.empty:
        continue
    event_errors = subset["event_time_absolute_error_s"].to_numpy(dtype=float)
    processing_errors = subset["processing_time_absolute_error_s"].to_numpy(dtype=float)
    reduction = processing_errors - event_errors
    rq1_rows.append(
        {
            "subset": subset_name,
            "rows": len(subset),
            "event_time_mae_s": float(np.mean(event_errors)),
            "processing_time_mae_s": float(np.mean(processing_errors)),
            "mae_reduction_s": float(np.mean(reduction)),
            "processing_time_median_absolute_error_s": float(np.median(processing_errors)),
            "processing_time_p95_absolute_error_s": float(np.percentile(processing_errors, 95)),
            "processing_time_max_absolute_error_s": float(np.max(processing_errors)),
            "h1_required_reduction_s": H1_REQUIRED_MAE_REDUCTION_S,
            "h1_supported_point_estimate": float(np.mean(reduction)) >= H1_REQUIRED_MAE_REDUCTION_S,
        }
    )

rq1_alignment_mae = pd.DataFrame(rq1_rows)
rq1_alignment_mae.to_csv(RQ1_MAE_OUT, index=False)

all_reductions = alignment_audit["processing_time_absolute_error_s"].to_numpy(dtype=float)
rq1_distribution = bootstrap_mean(
    all_reductions,
    BOOTSTRAP_ITERATIONS,
    RANDOM_STATE + 1,
)
rq1_ci_lower, rq1_ci_upper = percentile_interval(rq1_distribution)
rq1_point_reduction = require_float(
    rq1_alignment_mae.loc[
        rq1_alignment_mae["subset"].eq("all_events"), "mae_reduction_s"
    ].iloc[0],
    "H1 point MAE reduction",
)
rq1_bootstrap = pd.DataFrame(
    [
        {
            "comparison": "event_time_vs_processing_time",
            "point_mae_reduction_s": rq1_point_reduction,
            "bootstrap_mean_reduction_s": float(np.mean(rq1_distribution)),
            "bootstrap_ci_lower_95_s": rq1_ci_lower,
            "bootstrap_ci_upper_95_s": rq1_ci_upper,
            "probability_reduction_at_least_1s": float(
                np.mean(rq1_distribution >= H1_REQUIRED_MAE_REDUCTION_S)
            ),
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        }
    ]
)
rq1_bootstrap.to_csv(RQ1_BOOTSTRAP_OUT, index=False)

display(rq1_alignment_mae)
display(rq1_bootstrap)

## Alignment-Window Sensitivity

This section compares the frozen ±120-second primary window with the ±300-second and ±600-second sensitivity windows created by Notebook 03. It summarizes coverage and target stability; it does not retrain the models.

In [ ]:
require_columns(
    window_features,
    ["event_id", "window_name", "window_half_width_s", "nf3_flow_count"],
    "window features",
)
require_columns(
    window_targets,
    ["event_id", "window_name", "cyber_attack_window_label"],
    "window targets",
)

window_features["event_id"] = window_features["event_id"].astype("string")
window_targets["event_id"] = window_targets["event_id"].astype("string")
window_analysis = window_features.merge(
    window_targets[["event_id", "window_name", "cyber_attack_window_label"]],
    on=["event_id", "window_name"],
    how="inner",
    validate="one_to_one",
)
window_analysis["window_half_width_s"] = pd.to_numeric(
    window_analysis["window_half_width_s"], errors="raise"
).astype("int64")
window_analysis["nf3_flow_count"] = pd.to_numeric(
    window_analysis["nf3_flow_count"], errors="coerce"
).fillna(0)
window_analysis["cyber_attack_window_label"] = pd.to_numeric(
    window_analysis["cyber_attack_window_label"], errors="coerce"
).fillna(0).astype("int8")

primary_rows = window_analysis.loc[
    window_analysis["window_half_width_s"].eq(PRIMARY_WINDOW_HALF_WIDTH_S),
    ["event_id", "cyber_attack_window_label"],
].rename(columns={"cyber_attack_window_label": "primary_target"})
if primary_rows.empty:
    raise ValueError(
        f"No ±{PRIMARY_WINDOW_HALF_WIDTH_S}-second primary window was found."
    )

window_analysis = window_analysis.merge(
    primary_rows,
    on="event_id",
    how="left",
    validate="many_to_one",
)

window_sensitivity_rows: list[dict[str, object]] = []
for half_width_value, group in window_analysis.groupby("window_half_width_s", sort=True):
    half_width_s = require_int(half_width_value, "window half-width")
    target = group["cyber_attack_window_label"].to_numpy(dtype=np.int8)
    primary_target = group["primary_target"].to_numpy(dtype=np.int8)
    positive_intersection = require_int(
        np.logical_and(target == 1, primary_target == 1).sum(),
        "positive intersection",
    )
    positive_union = require_int(
        np.logical_or(target == 1, primary_target == 1).sum(),
        "positive union",
    )
    window_sensitivity_rows.append(
        {
            "window_half_width_s": half_width_s,
            "window_name": require_text(group["window_name"].iloc[0], "window name"),
            "events": len(group),
            "events_with_flows": require_int(group["nf3_flow_count"].gt(0).sum(), "events with flows"),
            "flow_coverage_fraction": float(group["nf3_flow_count"].gt(0).mean()),
            "median_flow_count": float(group["nf3_flow_count"].median()),
            "mean_flow_count": float(group["nf3_flow_count"].mean()),
            "maximum_flow_count": require_int(group["nf3_flow_count"].max(), "maximum flow count"),
            "positive_targets": require_int(target.sum(), "positive targets"),
            "target_prevalence": float(target.mean()),
            "target_changes_vs_primary": require_int((target != primary_target).sum(), "target changes"),
            "positive_jaccard_vs_primary": (
                float(positive_intersection / positive_union) if positive_union else 1.0
            ),
            "boundary_truncated_events": (
                require_int(
                    pd.to_numeric(
                        group["window_boundary_truncated_flag"], errors="coerce"
                    ).fillna(0).sum(),
                    "boundary-truncated events",
                )
                if "window_boundary_truncated_flag" in group.columns
                else np.nan
            ),
        }
    )

window_sensitivity = pd.DataFrame(window_sensitivity_rows).sort_values("window_half_width_s")
window_sensitivity.to_csv(WINDOW_SENSITIVITY_OUT, index=False)
display(window_sensitivity)

## Frozen Model Selection and Common Test Set

The strongest tabular comparator is selected using **validation AUPRC only** among non-dummy fused models under the chronological split. The test set is then intersected across the selected tabular model, fused Transformer, full R-GCN, and R-GCN ablation so all paired comparisons use identical events.

In [ ]:
require_columns(
    tabular_results,
    [
        "split_strategy", "feature_set", "model", "validation_auprc", "test_auprc",
        "test_f1", "test_recall", "selected_threshold", "latency_p95_ms",
    ],
    "tabular results",
)
require_columns(
    transformer_results,
    [
        "feature_set", "model", "validation_auprc", "test_auprc", "test_f1",
        "test_recall", "selected_threshold", "latency_p95_ms",
    ],
    "Transformer results",
)
require_columns(
    rgcn_results,
    [
        "variant", "validation_auprc", "test_auprc", "test_f1", "test_recall",
        "selected_threshold", "local_latency_p95_ms",
    ],
    "R-GCN results",
)

eligible_tabular = tabular_results.loc[
    tabular_results["split_strategy"].eq("chronological_primary")
    & tabular_results["feature_set"].eq("fused")
    & ~tabular_results["model"].eq("dummy_prior")
].copy()
if eligible_tabular.empty:
    raise ValueError("No non-dummy chronological fused tabular result was found.")
eligible_tabular["validation_auprc"] = pd.to_numeric(
    eligible_tabular["validation_auprc"], errors="coerce"
)
selected_tabular_row = eligible_tabular.sort_values(
    ["validation_auprc", "model"],
    ascending=[False, True],
    kind="mergesort",
).iloc[0]
selected_tabular_model = require_text(selected_tabular_row["model"], "selected tabular model")

primary_transformer_feature_set = require_text(
    transformer_reference.get("feature_set", "fused"),
    "primary Transformer feature set",
)
primary_transformer_rows = transformer_results.loc[
    transformer_results["feature_set"].eq(primary_transformer_feature_set)
]
if len(primary_transformer_rows) != 1:
    raise ValueError(
        f"Expected one primary Transformer result for {primary_transformer_feature_set!r}; "
        f"found {len(primary_transformer_rows)}."
    )
selected_transformer_row = primary_transformer_rows.iloc[0]

primary_rgcn_variant = require_text(
    rgcn_reference.get("primary_variant", "full_lag_relation"),
    "primary R-GCN variant",
)
ablation_rgcn_variant = "lag_relation_ablation"
primary_rgcn_rows = rgcn_results.loc[rgcn_results["variant"].eq(primary_rgcn_variant)]
ablation_rgcn_rows = rgcn_results.loc[rgcn_results["variant"].eq(ablation_rgcn_variant)]
if len(primary_rgcn_rows) != 1 or len(ablation_rgcn_rows) != 1:
    raise ValueError("Expected one full and one ablated R-GCN result row.")
selected_rgcn_row = primary_rgcn_rows.iloc[0]
selected_ablation_row = ablation_rgcn_rows.iloc[0]


def standardize_prediction_frame(
    frame: pd.DataFrame,
    model_key: str,
    model_label: str,
    event_column: str,
    target_column: str,
    probability_column: str,
    threshold_column: str,
) -> pd.DataFrame:
    require_columns(
        frame,
        [event_column, target_column, probability_column, threshold_column],
        f"{model_label} predictions",
    )
    standardized = pd.DataFrame(
        {
            "model_key": model_key,
            "model_label": model_label,
            "event_id": frame[event_column].astype("string"),
            "target": pd.to_numeric(frame[target_column], errors="raise").astype("int8"),
            "probability": pd.to_numeric(frame[probability_column], errors="raise").astype(float),
            "selected_threshold": pd.to_numeric(frame[threshold_column], errors="raise").astype(float),
        }
    )
    if standardized["event_id"].duplicated().any():
        raise ValueError(f"{model_label} contains duplicate event predictions.")
    if not standardized["probability"].between(0, 1).all():
        raise ValueError(f"{model_label} contains probabilities outside [0, 1].")
    return standardized


tabular_selected_predictions = tabular_predictions.loc[
    tabular_predictions["split_strategy"].eq("chronological_primary")
    & tabular_predictions["feature_set"].eq("fused")
    & tabular_predictions["model"].eq(selected_tabular_model)
].copy()
transformer_selected_predictions = transformer_predictions.loc[
    transformer_predictions["split"].eq("test")
    & transformer_predictions["feature_set"].eq(primary_transformer_feature_set)
].copy()
rgcn_full_predictions = rgcn_predictions.loc[
    rgcn_predictions["split"].eq("test")
    & rgcn_predictions["variant"].eq(primary_rgcn_variant)
].copy()
rgcn_ablation_predictions = rgcn_predictions.loc[
    rgcn_predictions["split"].eq("test")
    & rgcn_predictions["variant"].eq(ablation_rgcn_variant)
].copy()

standardized_frames = [
    standardize_prediction_frame(
        tabular_selected_predictions,
        "tabular_fused_best",
        f"Fused tabular — {selected_tabular_model}",
        "event_id", "model_target", "predicted_probability", "selected_threshold",
    ),
    standardize_prediction_frame(
        transformer_selected_predictions,
        "temporal_transformer_fused",
        "Temporal Transformer — fused",
        "event_id", "model_target", "predicted_probability", "selected_threshold",
    ),
    standardize_prediction_frame(
        rgcn_full_predictions,
        "rgcn_full_lag_relation",
        "R-GCN — full lag/relation",
        "event_id", "model_target", "probability", "selected_threshold",
    ),
    standardize_prediction_frame(
        rgcn_ablation_predictions,
        "rgcn_lag_relation_ablation",
        "R-GCN — lag/relation ablation",
        "event_id", "model_target", "probability", "selected_threshold",
    ),
]

common_event_ids = set(standardized_frames[0]["event_id"])
for standardized in standardized_frames[1:]:
    common_event_ids &= set(standardized["event_id"])
if not common_event_ids:
    raise ValueError("The selected models have no common test events.")

paired_prediction_bank = pd.concat(
    [
        standardized.loc[standardized["event_id"].isin(common_event_ids)].copy()
        for standardized in standardized_frames
    ],
    ignore_index=True,
)
paired_prediction_bank = paired_prediction_bank.sort_values(
    ["model_key", "event_id"], kind="mergesort"
).reset_index(drop=True)

per_event_target_counts = paired_prediction_bank.groupby("event_id")["target"].nunique()
if not per_event_target_counts.eq(1).all():
    raise AssertionError("Model prediction files disagree on one or more test targets.")
expected_common_rows = len(common_event_ids) * len(standardized_frames)
if len(paired_prediction_bank) != expected_common_rows:
    raise AssertionError(
        f"Expected {expected_common_rows:,} paired rows; found {len(paired_prediction_bank):,}."
    )

paired_prediction_bank.to_csv(PAIRED_PREDICTIONS_OUT, index=False)

selection_summary = pd.DataFrame(
    [
        {
            "role": "selected_tabular",
            "selection_basis": "highest validation AUPRC among chronological fused non-dummy models",
            "selection": selected_tabular_model,
        },
        {
            "role": "primary_transformer",
            "selection_basis": "frozen Notebook 05 primary feature set",
            "selection": primary_transformer_feature_set,
        },
        {
            "role": "primary_rgcn",
            "selection_basis": "frozen Notebook 06 primary variant",
            "selection": primary_rgcn_variant,
        },
        {
            "role": "rgcn_ablation",
            "selection_basis": "frozen Notebook 06 ablation variant",
            "selection": ablation_rgcn_variant,
        },
    ]
)
display(selection_summary)
print(f"Common paired test events: {len(common_event_ids):,}")

In [ ]:
model_comparison_rows: list[dict[str, object]] = []


def add_result_row(
    model_key: str,
    model_label: str,
    family: str,
    row: pd.Series,
    latency_column: str,
) -> None:
    model_comparison_rows.append(
        {
            "model_key": model_key,
            "model_label": model_label,
            "model_family": family,
            "validation_auprc": require_float(row["validation_auprc"], f"{model_label} validation AUPRC"),
            "test_auprc": require_float(row["test_auprc"], f"{model_label} test AUPRC"),
            "test_roc_auc": optional_float(row.get("test_roc_auc")),
            "test_precision": optional_float(row.get("test_precision")),
            "test_recall": require_float(row["test_recall"], f"{model_label} test recall"),
            "test_f1": require_float(row["test_f1"], f"{model_label} test F1"),
            "test_balanced_accuracy": optional_float(row.get("test_balanced_accuracy")),
            "test_brier_score": optional_float(row.get("test_brier_score")),
            "selected_threshold": require_float(row["selected_threshold"], f"{model_label} threshold"),
            "p95_latency_ms": require_float(row[latency_column], f"{model_label} p95 latency"),
            "latency_measurement": (
                "single_event_predict_proba"
                if family == "tabular"
                else "raw_sequence_preprocessing_plus_forward"
                if family == "temporal_transformer"
                else "prepared_local_causal_subgraph_forward"
            ),
        }
    )


add_result_row(
    "tabular_fused_best",
    f"Fused tabular — {selected_tabular_model}",
    "tabular",
    selected_tabular_row,
    "latency_p95_ms",
)
add_result_row(
    "temporal_transformer_fused",
    "Temporal Transformer — fused",
    "temporal_transformer",
    selected_transformer_row,
    "latency_p95_ms",
)
add_result_row(
    "rgcn_full_lag_relation",
    "R-GCN — full lag/relation",
    "rgcn",
    selected_rgcn_row,
    "local_latency_p95_ms",
)
add_result_row(
    "rgcn_lag_relation_ablation",
    "R-GCN — lag/relation ablation",
    "rgcn",
    selected_ablation_row,
    "local_latency_p95_ms",
)

model_comparison = pd.DataFrame(model_comparison_rows)
model_comparison.to_csv(MODEL_COMPARISON_OUT, index=False)
display(model_comparison.sort_values("test_auprc", ascending=False))

## Paired Bootstrap Confidence Intervals

Bootstrap resampling is stratified by the binary target and uses the same sampled event positions for every model. This preserves pairing when estimating AUPRC differences.

In [ ]:
model_keys = model_comparison["model_key"].astype(str).tolist()
base_model_key = model_keys[0]
base_frame = paired_prediction_bank.loc[
    paired_prediction_bank["model_key"].eq(base_model_key)
].sort_values("event_id")
ordered_event_ids = base_frame["event_id"].astype(str).tolist()
test_target = base_frame["target"].to_numpy(dtype=np.int8)

probability_by_model: dict[str, np.ndarray] = {}
for model_key in model_keys:
    model_frame = paired_prediction_bank.loc[
        paired_prediction_bank["model_key"].eq(model_key)
    ].set_index("event_id").loc[ordered_event_ids]
    model_target = model_frame["target"].to_numpy(dtype=np.int8)
    if not np.array_equal(model_target, test_target):
        raise AssertionError(f"Target order mismatch for {model_key}.")
    probability_by_model[model_key] = model_frame["probability"].to_numpy(dtype=float)

model_bootstrap, auprc_distributions = bootstrap_model_auprc(
    test_target,
    probability_by_model,
    BOOTSTRAP_ITERATIONS,
    RANDOM_STATE + 2,
)
model_bootstrap = model_bootstrap.merge(
    model_comparison[["model_key", "model_label"]],
    on="model_key",
    how="left",
    validate="one_to_one",
)
model_bootstrap.to_csv(MODEL_BOOTSTRAP_OUT, index=False)

point_auprc = {
    model_key: safe_auprc(test_target, probability)
    for model_key, probability in probability_by_model.items()
}
paired_difference_rows = [
    paired_difference_summary(
        "rgcn_full_lag_relation",
        "temporal_transformer_fused",
        auprc_distributions,
        point_auprc,
        H2_REQUIRED_AUPRC_MARGIN,
        "H2: full R-GCN minus temporal Transformer",
    ),
    paired_difference_summary(
        "rgcn_full_lag_relation",
        "rgcn_lag_relation_ablation",
        auprc_distributions,
        point_auprc,
        H3_REQUIRED_AUPRC_GAIN,
        "H3: full R-GCN minus lag/relation ablation",
    ),
    paired_difference_summary(
        "temporal_transformer_fused",
        "tabular_fused_best",
        auprc_distributions,
        point_auprc,
        0.0,
        "Temporal Transformer minus strongest fused tabular baseline",
    ),
]
paired_differences = pd.DataFrame(paired_difference_rows)
paired_differences.to_csv(PAIRED_DIFFERENCES_OUT, index=False)

display(model_bootstrap.sort_values("point_test_auprc", ascending=False))
display(paired_differences)

## Subgroup Robustness

Frozen model thresholds are applied within descriptive subgroups. These analyses are diagnostic rather than separate hypothesis tests. Groups below the configured minimum size or without both target classes are omitted.

In [ ]:
metadata_columns = [
    "event_id",
    *(["source_type"] if "source_type" in model_data.columns else []),
    *(["stale_event_flag"] if "stale_event_flag" in model_data.columns else []),
    *(["prior_event_count"] if "prior_event_count" in model_data.columns else []),
    *(["nf3_flow_count"] if "nf3_flow_count" in model_data.columns else []),
]
subgroup_metadata = model_data[metadata_columns].copy()
subgroup_metadata["event_id"] = subgroup_metadata["event_id"].astype("string")
if "nf3_flow_count" in subgroup_metadata.columns:
    flow_count = pd.to_numeric(subgroup_metadata["nf3_flow_count"], errors="coerce").fillna(0)
    subgroup_metadata["nf3_flow_presence"] = np.where(flow_count.gt(0), "has_flows", "no_flows")
if "prior_event_count" in subgroup_metadata.columns:
    prior_count = pd.to_numeric(subgroup_metadata["prior_event_count"], errors="coerce").fillna(0)
    subgroup_metadata["recurrence_context"] = np.where(
        prior_count.gt(0), "prior_event_present", "first_observed_event"
    )
if "stale_event_flag" in subgroup_metadata.columns:
    stale = pd.to_numeric(subgroup_metadata["stale_event_flag"], errors="coerce").fillna(0)
    subgroup_metadata["timestamp_freshness"] = np.where(stale.eq(1), "stale", "non_stale")

subgroup_variables = [
    column
    for column in ["source_type", "nf3_flow_presence", "recurrence_context", "timestamp_freshness"]
    if column in subgroup_metadata.columns
]

subgroup_bank = paired_prediction_bank.merge(
    subgroup_metadata,
    on="event_id",
    how="left",
    validate="many_to_one",
)
subgroup_rows: list[dict[str, object]] = []
for model_key, model_frame in subgroup_bank.groupby("model_key", sort=False):
    normalized_model_key = require_text(model_key, "subgroup model key")
    model_label = require_text(model_frame["model_label"].iloc[0], "subgroup model label")
    for subgroup_variable in subgroup_variables:
        for subgroup_value, group in model_frame.groupby(subgroup_variable, dropna=False, sort=True):
            normalized_subgroup_value = "<missing>" if pd.isna(subgroup_value) else str(subgroup_value)
            if len(group) < MIN_SUBGROUP_ROWS or not has_both_classes(group["target"]):
                continue
            target = group["target"].to_numpy(dtype=np.int8)
            probability = group["probability"].to_numpy(dtype=float)
            threshold_values = group["selected_threshold"].dropna().unique()
            if len(threshold_values) != 1:
                raise ValueError(f"Non-unique frozen threshold in subgroup for {normalized_model_key}.")
            threshold = require_float(threshold_values[0], "subgroup frozen threshold")
            metrics = classification_metrics(target, probability, threshold)
            subgroup_rows.append(
                {
                    "model_key": normalized_model_key,
                    "model_label": model_label,
                    "subgroup_variable": subgroup_variable,
                    "subgroup_value": normalized_subgroup_value,
                    "rows": len(group),
                    "positives": require_int(target.sum(), "subgroup positives"),
                    "prevalence": float(target.mean()),
                    "frozen_threshold": threshold,
                    **metrics,
                }
            )

subgroup_performance = pd.DataFrame(subgroup_rows)
subgroup_performance.to_csv(SUBGROUP_OUT, index=False)
if subgroup_performance.empty:
    print("No subgroups met the minimum size and class-balance requirements.")
else:
    display(subgroup_performance.sort_values(["subgroup_variable", "model_key", "subgroup_value"]))

## Explainability Synthesis

Attribution magnitudes from different model families are not placed on a common numerical scale. The consolidated table preserves each method and reports within-source ranks only.

In [ ]:
explainability_frames: list[pd.DataFrame] = []

transformer_attribution = optional_tables["transformer_attribution"].copy()
if not transformer_attribution.empty:
    require_columns(
        transformer_attribution,
        ["transformed_feature", "mean_absolute_gradient_x_input"],
        "Transformer attribution",
    )
    transformer_attribution["importance"] = pd.to_numeric(
        transformer_attribution["mean_absolute_gradient_x_input"], errors="coerce"
    )
    transformer_attribution = transformer_attribution.sort_values(
        "importance", ascending=False, kind="mergesort"
    ).reset_index(drop=True)
    transformer_attribution["rank"] = np.arange(1, len(transformer_attribution) + 1)
    explainability_frames.append(
        pd.DataFrame(
            {
                "model_family": "temporal_transformer",
                "component_type": "transformed_feature",
                "component": transformer_attribution["transformed_feature"].astype(str),
                "importance_metric": "mean_absolute_gradient_times_input",
                "importance": transformer_attribution["importance"],
                "rank_within_source": transformer_attribution["rank"],
                "variant": "fused",
            }
        )
    )

rgcn_attribution = optional_tables["rgcn_attribution"].copy()
if not rgcn_attribution.empty:
    require_columns(
        rgcn_attribution,
        ["node_type", "feature", "mean_absolute_gradient_times_input", "rank_within_node_type"],
        "R-GCN attribution",
    )
    explainability_frames.append(
        pd.DataFrame(
            {
                "model_family": "rgcn",
                "component_type": rgcn_attribution["node_type"].astype(str) + "_node_feature",
                "component": rgcn_attribution["feature"].astype(str),
                "importance_metric": "mean_absolute_gradient_times_input",
                "importance": pd.to_numeric(
                    rgcn_attribution["mean_absolute_gradient_times_input"], errors="coerce"
                ),
                "rank_within_source": pd.to_numeric(
                    rgcn_attribution["rank_within_node_type"], errors="coerce"
                ),
                "variant": primary_rgcn_variant,
            }
        )
    )

relation_norms = optional_tables["rgcn_relation_norms"].copy()
if not relation_norms.empty:
    require_columns(
        relation_norms,
        ["variant", "layer_index", "relation", "weight_frobenius_norm"],
        "R-GCN relation norms",
    )
    relation_norms["importance"] = pd.to_numeric(
        relation_norms["weight_frobenius_norm"], errors="coerce"
    )
    relation_norms["rank"] = relation_norms.groupby(
        ["variant", "layer_index"]
    )["importance"].rank(method="first", ascending=False)
    explainability_frames.append(
        pd.DataFrame(
            {
                "model_family": "rgcn",
                "component_type": "relation_weight_layer_" + relation_norms["layer_index"].astype(str),
                "component": relation_norms["relation"].astype(str),
                "importance_metric": "linear_weight_frobenius_norm",
                "importance": relation_norms["importance"],
                "rank_within_source": relation_norms["rank"],
                "variant": relation_norms["variant"].astype(str),
            }
        )
    )

explainability_summary = (
    pd.concat(explainability_frames, ignore_index=True)
    if explainability_frames
    else pd.DataFrame(
        columns=[
            "model_family", "component_type", "component", "importance_metric",
            "importance", "rank_within_source", "variant",
        ]
    )
)
explainability_summary.to_csv(EXPLAINABILITY_OUT, index=False)

if explainability_summary.empty:
    print("No explainability artifacts were available.")
else:
    display(
        explainability_summary.sort_values(
            ["model_family", "component_type", "rank_within_source"]
        ).groupby(["model_family", "component_type"], group_keys=False).head(10)
    )

## Consolidated Hypothesis Decisions

The frozen point-estimate criteria determine support. Bootstrap intervals are reported as uncertainty diagnostics and do not retroactively change the preregistered thresholds.

In [ ]:
h2_row = paired_differences.loc[
    paired_differences["comparison"].str.startswith("H2:")
].iloc[0]
h3_row = paired_differences.loc[
    paired_differences["comparison"].str.startswith("H3:")
].iloc[0]
full_rgcn_latency_ms = require_float(
    model_comparison.loc[
        model_comparison["model_key"].eq("rgcn_full_lag_relation"), "p95_latency_ms"
    ].iloc[0],
    "full R-GCN p95 latency",
)

h1_supported = rq1_point_reduction >= H1_REQUIRED_MAE_REDUCTION_S
h2_margin = require_float(h2_row["point_auprc_difference"], "H2 margin")
h3_gain = require_float(h3_row["point_auprc_difference"], "H3 gain")
h2_supported = h2_margin >= H2_REQUIRED_AUPRC_MARGIN
h3_performance_pass = h3_gain >= H3_REQUIRED_AUPRC_GAIN
h3_latency_pass = full_rgcn_latency_ms <= H3_MAX_P95_LATENCY_MS
h3_supported = h3_performance_pass and h3_latency_pass

hypothesis_evaluation = pd.DataFrame(
    [
        {
            "hypothesis": "H1",
            "research_question": "RQ1",
            "criterion": "Event-time MAE reduction versus processing time >= 1.0 second",
            "observed_value": rq1_point_reduction,
            "required_value": H1_REQUIRED_MAE_REDUCTION_S,
            "bootstrap_ci_lower_95": require_float(
                rq1_bootstrap["bootstrap_ci_lower_95_s"].iloc[0], "H1 bootstrap lower CI"
            ),
            "bootstrap_ci_upper_95": require_float(
                rq1_bootstrap["bootstrap_ci_upper_95_s"].iloc[0], "H1 bootstrap upper CI"
            ),
            "latency_value_ms": np.nan,
            "latency_limit_ms": np.nan,
            "supported": h1_supported,
        },
        {
            "hypothesis": "H2",
            "research_question": "RQ2",
            "criterion": "Full R-GCN test AUPRC - temporal Transformer test AUPRC >= 0.03",
            "observed_value": h2_margin,
            "required_value": H2_REQUIRED_AUPRC_MARGIN,
            "bootstrap_ci_lower_95": optional_float(h2_row["bootstrap_ci_lower_95"]),
            "bootstrap_ci_upper_95": optional_float(h2_row["bootstrap_ci_upper_95"]),
            "latency_value_ms": np.nan,
            "latency_limit_ms": np.nan,
            "supported": h2_supported,
        },
        {
            "hypothesis": "H3",
            "research_question": "RQ3",
            "criterion": (
                "Full R-GCN test AUPRC - lag/relation ablation test AUPRC >= 0.05 "
                "and full local p95 latency <= 500 ms"
            ),
            "observed_value": h3_gain,
            "required_value": H3_REQUIRED_AUPRC_GAIN,
            "bootstrap_ci_lower_95": optional_float(h3_row["bootstrap_ci_lower_95"]),
            "bootstrap_ci_upper_95": optional_float(h3_row["bootstrap_ci_upper_95"]),
            "latency_value_ms": full_rgcn_latency_ms,
            "latency_limit_ms": H3_MAX_P95_LATENCY_MS,
            "supported": h3_supported,
        },
    ]
)
hypothesis_evaluation.to_csv(HYPOTHESES_OUT, index=False)
display(hypothesis_evaluation)

In [ ]:
# Dissertation-ready figures

plt.figure(figsize=(7, 5))
plt.bar(
    ["Event time", "Processing time"],
    [
        require_float(
            rq1_alignment_mae.loc[
                rq1_alignment_mae["subset"].eq("all_events"), "event_time_mae_s"
            ].iloc[0],
            "event-time MAE",
        ),
        require_float(
            rq1_alignment_mae.loc[
                rq1_alignment_mae["subset"].eq("all_events"), "processing_time_mae_s"
            ].iloc[0],
            "processing-time MAE",
        ),
    ],
)
plt.ylabel("Mean absolute timestamp error (seconds)")
plt.title("RQ1: Event-Time vs Processing-Time Timestamp Fidelity")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rq1_timestamp_mae_comparison.png", dpi=180)
plt.close()

plot_models = model_bootstrap.merge(
    model_comparison[["model_key", "model_label"]],
    on=["model_key", "model_label"],
    how="left",
)
plot_models = plot_models.sort_values("point_test_auprc", ascending=True)
error_lower = plot_models["point_test_auprc"] - plot_models["bootstrap_ci_lower_95"]
error_upper = plot_models["bootstrap_ci_upper_95"] - plot_models["point_test_auprc"]
plt.figure(figsize=(10, 6))
plt.barh(plot_models["model_label"], plot_models["point_test_auprc"])
plt.errorbar(
    plot_models["point_test_auprc"],
    np.arange(len(plot_models)),
    xerr=np.vstack([error_lower.to_numpy(dtype=float), error_upper.to_numpy(dtype=float)]),
    fmt="none",
    capsize=4,
)
plt.xlabel("Test AUPRC with paired bootstrap 95% CI")
plt.title("Held-Out Model Comparison")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_test_auprc_bootstrap_ci.png", dpi=180)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(
    window_sensitivity["window_half_width_s"],
    window_sensitivity["flow_coverage_fraction"],
    marker="o",
    label="Events with matched flows",
)
plt.plot(
    window_sensitivity["window_half_width_s"],
    window_sensitivity["target_prevalence"],
    marker="o",
    label="Positive-target prevalence",
)
plt.xlabel("Window half-width (seconds)")
plt.ylabel("Fraction")
plt.title("Alignment-Window Sensitivity")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "alignment_window_sensitivity.png", dpi=180)
plt.close()

plt.figure(figsize=(8, 6))
for model_key in model_keys:
    probability = probability_by_model[model_key]
    precision_values, recall_values, _ = precision_recall_curve(test_target, probability)
    label = require_text(
        model_comparison.loc[
            model_comparison["model_key"].eq(model_key), "model_label"
        ].iloc[0],
        "precision-recall model label",
    )
    plt.plot(recall_values, precision_values, label=label)
plt.axhline(float(np.mean(test_target)), linestyle="--", label="Test prevalence")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves on Common Held-Out Events")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "common_test_precision_recall_curves.png", dpi=180)
plt.close()

if not explainability_summary.empty:
    top_relations = explainability_summary.loc[
        explainability_summary["component_type"].str.startswith("relation_weight")
        & explainability_summary["variant"].eq(primary_rgcn_variant)
    ].sort_values("importance", ascending=False).head(15).sort_values("importance")
    if not top_relations.empty:
        plt.figure(figsize=(10, 7))
        plt.barh(
            top_relations["component_type"] + ": " + top_relations["component"],
            top_relations["importance"],
        )
        plt.xlabel("Relation linear-map Frobenius norm")
        plt.title("Full R-GCN Relation-Weight Diagnostics")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "rgcn_relation_weight_diagnostics.png", dpi=180)
        plt.close()

print(f"Saved {len(list(FIGURES_DIR.glob('*.png')))} figures to {FIGURES_DIR}")

In [ ]:
def support_word(value: object) -> str:
    return "supported" if bool(value) else "not supported"

best_model_row = model_comparison.sort_values("test_auprc", ascending=False).iloc[0]
best_model_label = require_text(best_model_row["model_label"], "best model label")
best_model_auprc = require_float(best_model_row["test_auprc"], "best model AUPRC")

h1_record = hypothesis_evaluation.loc[hypothesis_evaluation["hypothesis"].eq("H1")].iloc[0]
h2_record = hypothesis_evaluation.loc[hypothesis_evaluation["hypothesis"].eq("H2")].iloc[0]
h3_record = hypothesis_evaluation.loc[hypothesis_evaluation["hypothesis"].eq("H3")].iloc[0]

narrative = f"""# Notebook 07 Results Narrative

## Dataset and evaluation design

The consolidated analysis retained {len(model_data):,} candidate low-altitude point events. Model comparisons used {len(common_event_ids):,} common held-out events so all paired AUPRC differences were calculated on identical targets. The cyber labels and features originate from the controlled Notebook 03 aircraft-to-NF3 temporal mapping; they do not represent observed site-coincident attacks at PANC.

## RQ1 and H1

Processing-time timestamps had an MAE of {require_float(rq1_alignment_mae.loc[rq1_alignment_mae['subset'].eq('all_events'), 'processing_time_mae_s'].iloc[0], 'processing MAE'):.3f} seconds relative to aircraft-reported event time. Event-time timestamps had an MAE of 0 seconds by definition, yielding a reduction of {require_float(h1_record['observed_value'], 'H1 observed value'):.3f} seconds (95% bootstrap CI {require_float(h1_record['bootstrap_ci_lower_95'], 'H1 lower CI'):.3f} to {require_float(h1_record['bootstrap_ci_upper_95'], 'H1 upper CI'):.3f}). H1 was {support_word(h1_record['supported'])} under the preregistered 1.0-second criterion. This result evaluates timestamp fidelity, not real cross-domain clock synchronization.

## Model comparison

The strongest held-out point estimate was produced by **{best_model_label}** with test AUPRC {best_model_auprc:.4f}. The full R-GCN minus Transformer AUPRC difference was {require_float(h2_record['observed_value'], 'H2 observed value'):+.4f}; H2 was {support_word(h2_record['supported'])}. The full R-GCN minus lag/relation-ablation difference was {require_float(h3_record['observed_value'], 'H3 observed value'):+.4f}, and the full R-GCN local p95 latency was {require_float(h3_record['latency_value_ms'], 'H3 latency'):.3f} ms; H3 was {support_word(h3_record['supported'])}.

## Sensitivity and explainability

Window sensitivity was summarized for every Notebook 03 temporal window without retraining the models. Subgroup diagnostics retained frozen validation-selected thresholds and omitted groups that were too small or lacked both classes. Transformer and R-GCN attribution magnitudes were ranked within their own explanation methods and were not treated as directly comparable effect sizes.

## Interpretation boundary

The findings characterize predictive and explanatory behavior under a controlled multimodal alignment experiment. They do not show that any NF3 flow caused, accompanied, or was generated by a specific aircraft event.
"""
NARRATIVE_OUT.write_text(narrative, encoding="utf-8")
print(narrative)

In [ ]:
reference_payload = {
    "results_version": RESULTS_VERSION,
    "primary_model_selection": {
        "tabular_model": selected_tabular_model,
        "transformer_feature_set": primary_transformer_feature_set,
        "rgcn_primary_variant": primary_rgcn_variant,
        "rgcn_ablation_variant": ablation_rgcn_variant,
    },
    "common_test_events": len(common_event_ids),
    "hypotheses": json.loads(hypothesis_evaluation.to_json(orient="records")),
    "model_results": json.loads(model_comparison.to_json(orient="records")),
    "interpretation": (
        "Controlled cross-domain alignment performance; not evidence of causal or "
        "site-coincident aircraft/cyber incidents."
    ),
}
REFERENCE_OUT.write_text(
    json.dumps(reference_payload, indent=2, default=str),
    encoding="utf-8",
)

configuration_payload = {
    "results_version": RESULTS_VERSION,
    "random_state": RANDOM_STATE,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_policy": "paired stratified resampling of common held-out events",
    "minimum_subgroup_rows": MIN_SUBGROUP_ROWS,
    "primary_window_half_width_s": PRIMARY_WINDOW_HALF_WIDTH_S,
    "hypothesis_criteria": {
        "H1_required_mae_reduction_s": H1_REQUIRED_MAE_REDUCTION_S,
        "H2_required_auprc_margin": H2_REQUIRED_AUPRC_MARGIN,
        "H3_required_auprc_gain": H3_REQUIRED_AUPRC_GAIN,
        "H3_max_p95_latency_ms": H3_MAX_P95_LATENCY_MS,
    },
}
CONFIG_OUT.write_text(
    json.dumps(configuration_payload, indent=2, default=str),
    encoding="utf-8",
)

artifact_descriptions = {
    RQ1_MAE_OUT.name: "Event-time and processing-time timestamp MAE, descriptive errors, and H1 point decision.",
    RQ1_BOOTSTRAP_OUT.name: "Bootstrap uncertainty for the H1 MAE reduction.",
    WINDOW_SENSITIVITY_OUT.name: "Flow coverage and target stability across Notebook 03 temporal windows.",
    MODEL_COMPARISON_OUT.name: "Unified validation/test metrics and latency for the selected tabular, Transformer, and R-GCN models.",
    PAIRED_PREDICTIONS_OUT.name: "Standardized predictions on common held-out events for paired comparisons.",
    MODEL_BOOTSTRAP_OUT.name: "Paired stratified-bootstrap AUPRC intervals for each selected model.",
    PAIRED_DIFFERENCES_OUT.name: "Paired AUPRC differences, confidence intervals, and margin probabilities.",
    SUBGROUP_OUT.name: "Frozen-threshold descriptive subgroup metrics when sample and class requirements are met.",
    EXPLAINABILITY_OUT.name: "Within-method Transformer and R-GCN explanation rankings.",
    HYPOTHESES_OUT.name: "Consolidated H1–H3 criteria, observed values, uncertainty, latency, and support decisions.",
    INTEGRITY_OUT.name: "Structural checks applied before results consolidation.",
    REFERENCE_OUT.name: "Machine-readable frozen Notebook 07 results reference.",
    NARRATIVE_OUT.name: "Automatically generated dissertation-ready results narrative with interpretation caveats.",
    CONFIG_OUT.name: "Frozen Notebook 07 analysis configuration.",
    SUMMARY_OUT.name: "Provenance, environment, counts, checks, and output inventory.",
}

pd.DataFrame(
    [
        {"artifact": artifact, "artifact_role": description}
        for artifact, description in artifact_descriptions.items()
    ]
).to_csv(DATA_DICTIONARY_OUT, index=False)

input_paths = [
    FUSED_IN,
    WINDOW_FEATURES_IN,
    WINDOW_TARGETS_IN,
    SPLITS_IN,
    TABULAR_RESULTS_IN,
    TABULAR_PREDICTIONS_IN,
    TRANSFORMER_RESULTS_IN,
    TRANSFORMER_PREDICTIONS_IN,
    TRANSFORMER_REFERENCE_IN,
    RGCN_RESULTS_IN,
    RGCN_PREDICTIONS_IN,
    RGCN_REFERENCE_IN,
]
input_checksums = {
    path.relative_to(PROJECT_ROOT).as_posix(): sha256_file(path)
    for path in input_paths
}

summary_payload = {
    "definitions": configuration_payload,
    "interpretation": {
        "site_coincident_claim": False,
        "causal_claim": False,
        "statement": reference_payload["interpretation"],
    },
    "inputs": {
        "event_rows": len(model_data),
        "common_test_events": len(common_event_ids),
        "test_positives": require_int(test_target.sum(), "test positives"),
        "test_prevalence": float(test_target.mean()),
        "input_sha256": input_checksums,
        "alignment_mode": alignment_summary.get("alignment_version", "leakage_safe_nf3_source_block_partitioning"),
    },
    "outputs": {
        "model_rows": len(model_comparison),
        "paired_prediction_rows": len(paired_prediction_bank),
        "subgroup_rows": len(subgroup_performance),
        "explainability_rows": len(explainability_summary),
        "figure_files": len(list(FIGURES_DIR.glob("*.png"))),
        "hypotheses": json.loads(hypothesis_evaluation.to_json(orient="records")),
    },
    "environment": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "integrity_checks": json.loads(integrity_checks.to_json(orient="records")),
}
SUMMARY_OUT.write_text(
    json.dumps(summary_payload, indent=2, default=str),
    encoding="utf-8",
)

artifact_paths = [
    RQ1_MAE_OUT,
    RQ1_BOOTSTRAP_OUT,
    WINDOW_SENSITIVITY_OUT,
    MODEL_COMPARISON_OUT,
    PAIRED_PREDICTIONS_OUT,
    MODEL_BOOTSTRAP_OUT,
    PAIRED_DIFFERENCES_OUT,
    SUBGROUP_OUT,
    EXPLAINABILITY_OUT,
    HYPOTHESES_OUT,
    INTEGRITY_OUT,
    REFERENCE_OUT,
    SUMMARY_OUT,
    NARRATIVE_OUT,
    DATA_DICTIONARY_OUT,
    CONFIG_OUT,
    *sorted(FIGURES_DIR.glob("*.png")),
]
checksum_lines = [
    f"{sha256_file(path)}  {path.relative_to(OUTPUT_DIR).as_posix()}"
    for path in artifact_paths
    if path.is_file()
]
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

print("\nNotebook 07 completed successfully.")
print(f"H1: {support_word(h1_supported)}")
print(f"H2: {support_word(h2_supported)}")
print(f"H3: {support_word(h3_supported)}")
print(f"Output directory: {OUTPUT_DIR}")
print("Key output:", HYPOTHESES_OUT.name)